# 第94课 · 把六个月炼成五分钟——Aurora v1 全景 Demo 与作品自证

**目标**：打磨可展示 Demo——端到端故事线 + 验收清单，让「我真的做了、也真的懂」经得起追问。

🔗 **Aurora 连接**：
- `src/aurora/audio/` — Audio Core（FFT / STFT / Mel / MFCC，全手写）
- `src/aurora/mlops/`  — 训练循环与模型权重
- `src/aurora/speech/` — 语音识别模块
- `src/aurora/music/`  — 音乐特征与推荐（features.py / similarity.py）
- `src/aurora/llm/`    — LLM 集成层（kvcache / retrieve / sample）
- `src/aurora/rag/`    — 检索增强生成
- `tests/`            — 每个模块的基准测试（针对 numpy / 公式验证）


← **上一课**　[L93 · MLOps 基础](L93_mlops.ipynb)

> 上节课学习了 **MLOps 基础**：W&B 实验追踪、模型版本管理、Docker 打包与部署脚本。  
> 本课将探讨 **Aurora v1 全景 Demo**。

## 学习目标

完成本节后，你应该能够：

1. **用 WWHRL 框架**在 5 分钟内结构化介绍 Aurora，涵盖 What / Why / How / Results / Limits。
2. **通过 pytest 展示测试通过**，将代码基准（误差 < 1e-10）作为可验证的证据节点。
3. **填写完整的自证清单**（GitHub history / 博客 / 测试 / 录屏四层），每层至少有一个可展示的产出物。
4. **写出 6 条讲清楚每个里程碑的成果 bullet**（方法 + 功能 + 可查指标），对应 M1–M6 里程碑。


## 为什么需要"交付意识"

一个研究系统的价值，不仅取决于它能做什么，还取决于别人能不能在 5 分钟内看懂它能做什么。
Aurora v1 的核心卖点是**从零实现**（no API wrappers）：当有人问"你怎么做 FFT"（可能是面试官、同事，也可能是未来的你），答案不是"用 librosa"，
而是打开一个 notebook 直接运行。本节把 6 个月的成果组织成一条清晰的**证据链**，
并把演示流程、常见问答、成果 bullet 全部固化下来。


In [ ]:
import numpy as np


## 1. 证据链（Evidence Chain）

"我从零实现了 FFT"——这句话需要**可验证**的证据支撑，否则别人（甚至你自己）无法区分你是真懂还是只背过面经。
Aurora 的证据链由四层组成：

| 层级 | 形式 | 作用 |
|------|------|------|
| **GitHub history** | commit log + PR | 证明过程，不只是结果 |
| **博客文章** | `docs/current/blog/` Markdown | 证明理解深度，能讲清楚 |
| **测试结果** | `pytest tests/` 通过 | 证明代码正确，针对公式验证 |
| **Demo 录屏** | GIF / MP4 | 证明系统端到端可跑 |

每一层的核心逻辑：**越往下越难造假**。commit 时间戳不可改，测试对公式有误差界，
录屏显示真实推理时延。你的作品要按这四层准备，缺哪层就补哪层。


In [ ]:
# 证据链自检：打印各层的关键文件路径，确认它们存在
import os

AURORA_ROOT = os.path.expanduser("~/AURORA")

evidence = {
    "GitHub history": ".git/",
    "博客文章":       "docs/current/blog/",
    "测试目录":       "tests/",
    "ADR 决策记录":   "docs/current/adr/",
}

print("── 证据链自检 ──")
for label, rel in evidence.items():
    full = os.path.join(AURORA_ROOT, rel)
    exists = os.path.exists(full)
    status = "✅" if exists else "❌ 缺失"
    print(f"  {status}  {label:12s}  {full}")


## 2. 如何 Present AI 系统：WWHRL 框架

当你在 5 分钟内向别人（面试官、同事，或未来的自己）介绍系统时，对方要判断：这个人能不能独立推进一个 AI 系统？
结构化表达比堆细节有效得多。推荐用 **WWHRL**（What / Why / How / Results / Limits）：

```
What    — 系统做什么，一句话
Why     — 为什么从头实现而不用 API（研究诚信 + 真正理解）
How     — 关键算法，能展开讲（FFT → STFT → Mel → MFCC → 模型）
Results — 可量化指标（WER、延迟、测试通过率）
Limits  — 诚实说明边界（单说话人、小词表、非实时）
```

**陷阱**：把大量时间花在 What 上，跳过 Why 和 Limits。听众最关心 Why（判断思维深度）
和 Limits（判断自我认知）。Results 提供可信度锚点，但没有 Why 的 Results 是空的。

In [ ]:
# WWHRL 模板：打印 Aurora v1 的标准介绍稿（可直接背诵或改写）
wwhr = {
    "What": (
        "Aurora 是一个从零实现的音频 AI 系统，"
        "覆盖特征提取（MFCC）、语音识别（ASR）、推荐和 Agent 四个模块。"
    ),
    "Why": (
        "所有核心算法（FFT、Mel 滤波器组、DCT）全部手写，不调用 librosa / scipy.signal。"
        "目的是证明对 DSP 数学的第一性原理理解，而非 API 调用能力。"
    ),
    "How": (
        "FFT 用 Cooley-Tukey 蝶形分解实现，O(N log N)；"
        "Mel 滤波器组按 HTK 公式手动构造三角滤波器；"
        "MFCC 在 Mel 能量上做手写 DCT-II；"
        "训练循环用纯 NumPy（或轻量 PyTorch）实现梯度下降。"
    ),
    "Results": (
        "所有模块通过与 numpy.fft / 参考公式的误差测试（< 1e-10）；"
        "端到端 ASR demo 在 LibriSpeech clean 子集 [待填入：运行 scripts/eval_asr.py 后记录 WER 数值]。"
    ),
    "Limits": (
        "当前仅支持单说话人、16kHz 采样；"
        "推理非实时（无流式解码）；"
        "模型未在大规模数据上调优，WER 高于工业系统。"
    ),
}

print("── Aurora v1 WWHR 介绍稿 ──\n")
for key, val in wwhr.items():
    print(f"[{key}]\n  {val}\n")

## 3. 简历摘录：每个月份 → 一个 Bullet

简历上的 AI 系统 bullet 格式：

```
[方法] 实现 [功能]，[指标/对比]
```

例子：

```
• 手写 Cooley-Tukey FFT（O(N log N)），误差 < 1e-10（对比 numpy.fft）
• 构造 80-channel Mel 滤波器组 + DCT-II，端到端 MFCC 流水线（pipeline）无第三方 DSP 库
• 训练轻量 ASR 模型，LibriSpeech clean WER < X%，推理延迟 < Y ms（CPU）
• 实现基于 MFCC 相似度的音频推荐引擎，top-5 精度 Z%
• 搭建 Agent 层，自动调度特征提取 → 推理 → 结果聚合流水线
```

每条 bullet 的写法要点：
- **动词精确**：「手写」「构造」「实现」「搭建」，不用「使用」「基于」
- **指标可查**：误差界来自测试日志，WER 来自评测脚本
- **对比锚点**：有对比（vs numpy.fft、vs librosa）才有说服力


In [ ]:
# 简历 bullet 生成器：按月份输出当前已完成的 bullet
bullets = [
    ("M1", "Audio Core",  "手写 Cooley-Tukey FFT（O(N log N)），数值误差 < 1e-10（对比 numpy.fft）"),
    ("M2", "STFT / Mel",  "实现 STFT + 80-channel Mel 滤波器组（HTK 公式），无 librosa / scipy.signal"),
    ("M3", "MFCC",        "构造手写 DCT-II，完整 MFCC 流水线通过公式基准测试"),
    ("M4", "训练循环",    "从零搭建训练循环，损失曲线收敛，模型权重可复现"),
    ("M5", "ASR",         "端到端语音识别模块，输入音频 → 文本，[待填入：运行 scripts/eval_asr.py 后记录 WER 数值]"),
    ("M6", "Agent/Demo",  "Agent 调度层 + v1 Demo，覆盖特征提取→推理→推荐全链路"),
]

print("── 简历 Bullet（按月） ──\n")
for month, module, text in bullets:
    print(f"  [{month}] {module}")
    print(f"    • {text}")
    print()

# 检查：bullet 数量应等于里程碑数量
assert len(bullets) == 6, f"期望 6 条 bullet，实际 {len(bullets)}"
print("✅ 6 条 bullet 均已填写")

## 4. HuggingFace 生态与 Aurora 的定位

Aurora 的"no API wrappers"原则针对**核心 DSP 算法**（FFT / Mel / MFCC）。
对于超出手写范围的大模型，**了解并会用 HuggingFace** 是研究工程师的必备技能。

### HuggingFace `transformers` 核心模式

```python
from transformers import pipeline, AutoModel, AutoTokenizer

# 1. pipeline：最快上手，一行推理
asr = pipeline("automatic-speech-recognition", model="openai/whisper-small")
result = asr("audio.wav")  # → {"text": "..."}

# 2. AutoModel：需要访问 hidden states 或 embedding
model = AutoModel.from_pretrained("laion/larger_clap_general")
tokenizer = AutoTokenizer.from_pretrained("laion/larger_clap_general")

# 3. from_pretrained 参数控制精度和设备
model = AutoModel.from_pretrained(
    "facebook/musicgen-small",
    torch_dtype=torch.float16,   # 半精度省显存
    device_map="auto",           # 自动分配 CPU/GPU
)
```

### Aurora 与 HuggingFace 的分工

| 模块 | Aurora 自己实现 | 通过 HuggingFace 使用 |
|------|----------------|----------------------|
| 特征提取 | FFT / STFT / Mel / MFCC（from scratch） | — |
| KWS | 简单 CNN + 自训练 | — |
| ASR | CTC 前向算法（from scratch） | Whisper 推理 / 微调 |
| 音乐 Embedding | MusicEncoder + Triplet/NT-Xent | CLAP（大规模版本） |
| 音乐生成 | EnCodec 原理理解 | MusicGen（直接调用） |
| LLM / RAG | TF-IDF / 手写 Transformer（原理） | Claude / GPT API（应用） |

**关键判断力**：能说清楚"我实现了 X，因为想真正理解它；我用 HuggingFace 的 Y，因为从零实现 Y 需要 1 年和 10 万 GPU 小时"——这才是一个成熟构筑者的判断力，而不是"全都自己写"或"全都 import"。

In [ ]:
import numpy as np
import os

# ── Aurora 系统能力自查（可执行验证，替代静态表格）──────────────────────────────
# 注意判定口径：本矩阵按"是否已提炼进 src/"（对应文件存在与否）判定 ✅/🔲。
# CTC 前向算法与 asyncio 流水线已分别在 L68-69 / L92 的 notebook 里手写验证过，
# 只是尚未搬进 src/，所以这里仍显示 🔲 待实现。
AURORA_ROOT = os.path.expanduser("~/AURORA")

capabilities = [
    # (能力描述, 证明方式, 模块路径)
    ("FFT O(N log N)",          "误差 < 1e-10 vs numpy.fft",    "src/aurora/audio/transforms.py"),
    ("STFT 加窗分帧",           "形状、幅度与 scipy 对齐",       "src/aurora/audio/stft.py"),
    ("Mel 滤波器组 (80ch)",     "HTK 公式，能量对齐 librosa",    "src/aurora/audio/mel.py"),
    ("手写 DCT-II",             "误差 < 1e-12 vs scipy.fft.dct", "src/aurora/audio/transforms.py"),
    ("MFCC 完整流水线",         "与 librosa.feature.mfcc 对齐",  "src/aurora/audio/mfcc.py"),
    ("KWS CNN 分类器",          "计划中 — 尚未实现",              "src/aurora/mlops/kws.py (planned)"),
    ("CTC 前向算法",            "路径概率手推验证",              "src/aurora/speech/ctc.py"),
    ("Levenshtein / WER",       "与 jiwer 精确匹配",             "src/aurora/speech/metrics.py"),
    ("MusicEncoder NT-Xent",    "t-SNE 三类聚类分离",            "src/aurora/music/embed.py"),
    ("余弦 k-NN 推荐",          "Precision@5 > 0.5",             "src/aurora/music/similarity.py"),
    ("TF-IDF RAG",              "无 faiss / sentence-transformers","src/aurora/llm/retrieve.py"),
    ("asyncio 流水线",          "并行 ASR+RAG，p95 < 3s",        "src/aurora/realtime/runner.py"),
]

print("── Aurora v1 系统能力矩阵 ──\n")
print(f"{'能力':30s}  {'证明方式':35s}  {'状态'}")
print("─" * 85)
for cap, proof, path in capabilities:
    full = os.path.join(AURORA_ROOT, path)
    exists = os.path.exists(full)
    status = "✅" if exists else "🔲 待实现"
    print(f"{cap:30s}  {proof:35s}  {status}")

print()
done = sum(1 for _, _, p in capabilities if os.path.exists(os.path.join(AURORA_ROOT, p)))
print(f"已实现 {done}/{len(capabilities)} 个核心能力")
print("（判定口径：已提炼进 src/。CTC / asyncio 流水线已在 L68-69 / L92 手写验证，尚未搬进 src/）")

# ── HuggingFace 模型速查表（Aurora 依赖的外部大模型）────────────────────────────
print("\n── Aurora 依赖的 HuggingFace 模型 ──\n")
hf_models = [
    ("openai/whisper-small",          "ASR 推理 / 微调",        "~460MB", "transformers.pipeline"),
    ("openai/whisper-tiny",           "ASR 快速原型",           "~150MB", "transformers.pipeline"),
    ("laion/larger_clap_general",     "音频-文字对比 embedding", "~900MB", "ClapModel"),
    ("facebook/musicgen-small",       "文字→音乐生成",          "~1.5GB", "MusicgenForConditionalGeneration"),
    ("facebook/encodec_32khz",        "音频 tokenization",      "~100MB", "EncodecModel"),
    ("google/flan-t5-small",          "轻量文字编码（RAG用）",   "~300MB", "T5EncoderModel"),
]
print(f"{'模型 ID':42s}  {'用途':25s}  {'大小':8s}  {'HuggingFace 类'}")
print("─" * 100)
for model_id, use, size, cls in hf_models:
    print(f"{model_id:42s}  {use:25s}  {size:8s}  {cls}")

print("\n# 通用加载模板:")
print("from transformers import AutoModel, AutoProcessor")
print('model = AutoModel.from_pretrained("<model_id>", device_map="auto")')

## 参数实验：5 分钟演示排练 + 常见 Q&A

### 5 分钟演示流程

| 时间 | 内容 | 关键操作 |
|------|------|---------|
| 0:00–0:30 | 一句话介绍 Aurora | 说 WWHRL 中的 What + Why |
| 0:30–1:30 | 运行 FFT notebook  | 跑 `notebooks/5_audio_dsp/L42_visual_fft.ipynb`（🎨 FFT 图形化），展示蝶形（butterfly）图 |
| 1:30–2:30 | 运行 MFCC 流水线   | 跑 `notebooks/5_audio_dsp/L53_visual_mfcc.ipynb`（🎨 MFCC 图形化），展示 MFCC 热图 |
| 2:30–3:30 | 运行 ASR Demo      | 跑 `notebooks/7_asr/L75_visual_asr.ipynb`（🎨 ASR 流水线图形化），展示波形 → token → 文本 |
| 3:30–4:30 | 展示测试结果       | `pytest tests/ -v`，全绿 |
| 4:30–5:00 | Limits + Q&A       | 主动说限制，邀请追问 |

**预备常见问题 10 个**（别人会问，你自己也该能答；附答案方向，详见下方 code 格）：
1. 为什么不用 librosa？
2. FFT 的时间复杂度是多少，你怎么验证的？
3. Mel 滤波器组的频率间距为什么不是线性的？
4. DCT-II 和 DFT 有什么区别？
5. MFCC 的第 0 个系数代表什么？
6. 你的模型在噪声环境下表现如何？
7. 如何扩展到流式（实时）推理？
8. 推荐模块用的是什么相似度度量？
9. Agent 层怎么处理失败重试？
10. 如果给你三个月，你最优先改进什么？

In [ ]:
# 常见 Q&A 打印：运行一遍，确认答案都在脑子里
qa = [
    (
        "为什么不用 librosa？",
        "librosa.stft / librosa.feature.mfcc 是黑盒——你无法解释内部实现。"
        "手写逼着你理解每一步数学；被追问时能展开讲，不是只能说'调了个函数'。"
    ),
    (
        "FFT 的时间复杂度是多少，你怎么验证的？",
        "O(N log N)，Cooley-Tukey 每级做 N/2 次蝶形，共 log₂N 级。"
        "验证方式：对不同 N 计时，log-log 图斜率应约等于 1（log N 项），"
        "同时与 numpy.fft 比对数值误差 < 1e-10。"
    ),
    (
        "Mel 滤波器组的频率间距为什么不是线性的？",
        "人耳对低频分辨率高、对高频分辨率低，Mel 尺度模拟这一特性。"
        "公式 `mel = 2595 * log10(1 + hz/700)`，低频段 Mel 间距小、Hz 间距也小，"
        "所以滤波器在低频密集、高频稀疏。"
    ),
    (
        "DCT-II 和 DFT 有什么区别？",
        "DFT 假设信号周期延拓，会在边界产生不连续；DCT-II 假设偶对称延拓，"
        "边界连续，能量更集中在低频系数，适合压缩和特征提取。"
        "MFCC 用 DCT-II 是因为 Mel 能量之间高度相关，DCT 去相关后系数更独立。"
    ),
    (
        "MFCC 的第 0 个系数代表什么？",
        "C0 是所有 Mel 频带能量的加权和，近似于对数总能量（音量）。"
        "实践中常被丢弃或替换为对数帧能量，因为它对信道差异敏感。"
    ),
    (
        "你的模型在噪声环境下表现如何？",
        "当前未做噪声增强，在 SNR < 10dB 时 WER 显著上升。"
        "改进方向：数据增强（加噪/混响）或前端谱减法。这是已知 Limit。"
    ),
    (
        "如何扩展到流式（实时）推理？",
        "STFT 需要改成滑动窗口、每帧增量处理；模型换成流式解码（CTC greedy 或 beam）。"
        "当前的批量推理框架不支持，需要重构推理入口。"
    ),
    (
        "推荐模块用的是什么相似度度量？",
        "MFCC 向量之间的余弦相似度。公式：`cos(a,b) = dot(a,b) / (||a|| * ||b||)`。"
        "候选方向：用 DTW 做时序对齐，或加 FAISS 做近似最近邻加速。"
    ),
    (
        "Agent 层怎么处理失败重试？",
        "当前是简单的线性 pipeline，无重试逻辑——这是已知 Limit。"
        "生产级做法：每个步骤包装成 Task，失败后指数退避重试，超过阈值告警。"
    ),
    (
        "如果给你三个月，你最优先改进什么？",
        "流式推理 > 噪声鲁棒性 > 大词表语言模型融合。"
        "流式是最大工程缺口，噪声鲁棒性对实用场景影响最直接，"
        "语言模型融合（shallow fusion）能快速拉低 WER。"
    ),
]

print("── 常见 Q&A 速查 ──\n")
for i, (q, a) in enumerate(qa, 1):
    print(f"Q{i:02d}: {q}")
    print(f"  A:  {a}")
    print()

assert len(qa) == 10
print("✅ 10 道题全部准备完毕")


## 5. GitHub 主页：第一印象发生在代码仓库

别人（面试官、潜在合作者，甚至几个月后的你自己）打开你的项目，第一件事通常是打开 GitHub。
以下清单帮助你在 30 秒内给出正面印象：

- **Pinned repository**：把 `aurora` 置顶，描述写清楚 "what + why from scratch"
- **README 首屏**：30 秒能看懂这个项目做什么，有一条可以运行的命令
- **Commit 消息**：每条都是人话，例如 `feat(fft): implement Cooley-Tukey butterfly`
- **语言统计**：Python 主导（Aurora 自然满足）

`aurora` README 推荐首屏结构：
```
# Aurora · 音频 AI 从零实现
> FFT / STFT / Mel / MFCC / CTC——无 API 包装，纯算法实现

## 快速体验
\`\`\`bash
make install && python scripts/demo_audio.py
\`\`\`
```

In [ ]:
readme_checklist = {
    "一句话介绍（what + why from scratch）": None,  # True/False
    "有可运行的快速启动命令":                None,
    "核心能力表格（含误差/精度数字）":       None,
    "有至少 1 张截图或终端输出示例":         None,
    "测试 badge 或 CI 状态":               None,
    "License 已声明":                      None,
}
done = sum(1 for v in readme_checklist.values() if v is True)
print(f"README 自检：{done}/{len(readme_checklist)}")
for k, v in readme_checklist.items():
    icon = "✅" if v is True else ("❌" if v is False else "⬜")
    print(f"  {icon} {k}")

## 6. 技术博客：一篇文章 > 十个 commit

Aurora 天然适合写技术博客——每个从零实现的模块都是一个天然的故事：
为什么手写、踩了什么坑、结果和参考实现差多少。

| 文章类型 | 示例标题 |
|---|---|
| 推导记录 | "FFT 蝶形算法：从 DFT 定义到 O(N log N) 的完整推导" |
| 实现对比 | "手写 MFCC vs librosa：误差在哪里，为什么这样设计" |
| 踩坑记录 | "CTC 前向算法的三个边界条件——调了两天才找到" |
| 论文笔记 | "Whisper 精读：弱监督数据如何训练出强 ASR" |

发布平台：知乎（中文受众）/ Medium（英文）/ GitHub Pages（完全自控）

In [ ]:
first_blog = {
    "title":         None,   # 第一篇想写什么？
    "platform":      None,   # 发在哪里？
    "core_message":  None,   # 读者能学到什么？（一句话）
    "deadline":      None,   # 具体日期，例: '2026-07-15'
}
if all(v is not None for v in first_blog.values()):
    print("博客计划已就绪：")
    for k, v in first_blog.items():
        print(f"  {k}: {v}")
else:
    empty = [k for k, v in first_blog.items() if v is None]
    print(f"还有 {len(empty)} 项未填：{empty}")

## 本课收束

本节没有新的编码任务，核心输出是**三件可交付物**：自证清单报告、WWHR 介绍稿、成果 bullet 清单。
整个 Aurora 系统以 `src/aurora/audio/` 为基础、`tests/` 为验证层，形成从 FFT 到 Agent 的完整证据链。
真正的竞争力不在于指标高低，而在于能逐层展开 Why / How / Limits——无论对面是面试官、同事还是未来的自己，这三个函数名都比任何 WER 数字更有说服力。
下一步：录制 Demo 录屏、更新 `docs/current/blog/` 里的 M6 博客文章，把 Aurora v1 打包成可分享的 GitHub Release。


---

→ **下一课**　[L95 · 研究论文入门](L95_research_papers.ipynb)

> 下节课将学习 **研究论文入门**：三遍阅读法、论文结构写作、投稿流程与学术合作。